**Training of RTDETRL (Fine tuned, Base) Model and RTDETRL (HYBRID MobileSAM_CLIP)**

In [ ]:
!pip install ultralytics roboflow optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.6/324.6 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 140.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 11.3 MB/s eta 0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.2
    Uninstalling typer-0.27.2:
      Successfully uninstalled typer-0.27.2


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from roboflow import Roboflow
from pathlib import Path

ROBOFLOW_API_KEY = "qtnFdbzEhlnqVXaWWOt5"
WORKSPACE_ID = "hybridtunedmodeltraining"      # e.g. "my-workspace"
PROJECT_ID   = "finaldatasetrevised"        # e.g. "uav-cars"
VERSION_NUM  = 1                     # your dataset version number

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE_ID).project(PROJECT_ID)
dataset = project.version(VERSION_NUM).download("yolov8")  # creates a local folder

DATASET_DIR = Path(dataset.location)  # path to downloaded dataset folder
print("Downloaded to:", DATASET_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to FinalDataSetRevised-1 in yolov8:: 100%|██████████| 29786/29786 [00:12<00:00, 2368.35it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Downloaded to: /content/FinalDataSetRevised-1


RTDETRL (fine tuned)

In [ ]:
from ultralytics import RTDETR

In [ ]:
model = RTDETR("/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Fine_tuned_50epochs_trained_models/results_RTDETRL_FineTuned/run1/weights/best.pt")
model

RTDETR(
  (model): RTDETRDetectionModel(
    (model): Sequential(
      (0): HGStem(
        (stem1): Conv(
          (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): ReLU(inplace=True)
        )
        (stem2a): Conv(
          (conv): Conv2d(32, 16, kernel_size=(2, 2), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): ReLU(inplace=True)
        )
        (stem2b): Conv(
          (conv): Conv2d(16, 32, kernel_size=(2, 2), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): ReLU(inplace=True)
        )
        (stem3): Conv(
          (conv): Conv2d(64, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001

Set best params in below cell

In [ ]:
best_params = {
        "optimizer": "SGD",
        "lr0": 0.0007128723381029613,
        "lrf": 0.08815332252414224,
        "momentum": 0.9378893757528532,
        "weight_decay": 8.397158031058397e-05,
        "warmup_epochs": 2,
        "box": 5.027503537458358,
        "cls": 0.4767066612491639,
        "dfl": 1.8268886940573972
    }

In [ ]:
model.train(data="/content/FinalDataSetRevised-1/data.yaml",
            epochs=50, imgsz=640, batch=16,
            amp=True,
            project="/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Five_class_trained_models/results_5Classes_RTDETRL_Fine_tuned_50epochs_new", #change path here
            name="run1", exist_ok=True, **best_params)

Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=5.027503537458358, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.4767066612491639, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/FinalDataSetRevised-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.8268886940573972, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0007128723381029613, lrf=0.08815332252414224, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       1/50      12.6G      2.323     0.1812     0.6189         96        640: 100% ━━━━━━━━━━━━ 745/745 3.2it/s 3:54
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 3.6it/s 13.0s
                   all       1489      18279      0.816    0.00695    0.00238   0.000467

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      13.3G      2.322      0.103     0.4249        280        640: 0% ──────────── 0/745  0.3s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       2/50      13.3G      1.574     0.3265     0.2244        101        640: 100% ━━━━━━━━━━━━ 745/745 3.4it/s 3:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 5.8it/s 8.2s
                   all       1489      18279       0.96      0.157      0.155     0.0949

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      13.4G     0.7154     0.6419    0.06306        277        640: 0% ──────────── 0/745  0.3s

...

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      49/50      13.4G     0.2345     0.3643    0.02105         66        640: 100% ━━━━━━━━━━━━ 745/745 3.4it/s 3:37
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 5.7it/s 8.3s
                   all       1489      18279      0.974      0.554      0.605      0.509

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      50/50      13.4G     0.3106     0.4494    0.02145        156        640: 0% ──────────── 0/745  0.3s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      50/50      13.4G     0.2356     0.3641    0.02097         57        640: 100% ━━━━━━━━━━━━ 745/745 3.4it/s 3:37
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 5.7it/s 8.2s
                   all       1489      18279      0.974      0.553      0.607      0.511

50 epochs completed in 3.183 hours.
Optimizer stripped from /content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/results_5Classes_RTDETRL_50epochs/run1/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/results_5Classes_RTDETRL_50epochs/run1/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/results_5Classes_RTDETRL_50epochs/run1/weights/best.pt...
Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
rt-detr-l summary (fused): 315 layers, 31,994,015 parameters, 0 gradients, 105

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7abfd4ce7ee0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

TRAINING ON BASE MODEL

In [ ]:
from ultralytics import RTDETR
model = RTDETR("/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Base_50epoch_trained_models/results_RTDETRL_50epochs/run1/weights/best.pt")
model

RTDETR(
  (model): RTDETRDetectionModel(
    (model): Sequential(
      (0): HGStem(
        (stem1): Conv(
          (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): ReLU(inplace=True)
        )
        (stem2a): Conv(
          (conv): Conv2d(32, 16, kernel_size=(2, 2), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): ReLU(inplace=True)
        )
        (stem2b): Conv(
          (conv): Conv2d(16, 32, kernel_size=(2, 2), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): ReLU(inplace=True)
        )
        (stem3): Conv(
          (conv): Conv2d(64, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001

In [ ]:
model.train(data="/content/FinalDataSetRevised-1/data.yaml",
            epochs=50, imgsz=640, batch=16,
            amp=True,
            project="/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Five_class_trained_models/Incomplete_results_5Classes_Base_RTDETRL_new", #change path here
            name="run1", exist_ok=True)

Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/FinalDataSetRevised-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/results_RTDETRL_50epochs/run1

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       1/50        13G     0.3139     0.6329    0.03499         96        640: 100% ━━━━━━━━━━━━ 745/745 3.3it/s 3:43
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 4.3it/s 10.9s
                   all       1489      18279      0.886      0.922      0.914      0.799

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      13.8G     0.2442     0.3038    0.01498        280        640: 0% ──────────── 0/745  0.3s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       2/50      13.8G     0.2177     0.2815    0.01569        101        640: 100% ━━━━━━━━━━━━ 745/745 3.4it/s 3:40
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 5.6it/s 8.4s
                   all       1489      18279      0.967      0.946      0.973      0.839

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      13.9G     0.2349      0.297    0.01327        277        640: 0% ──────────── 0/745  0.3s

...

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      10/50      13.9G        nan        nan        nan        112        640: 100% ━━━━━━━━━━━━ 745/745 3.4it/s 3:39
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 7.1it/s 6.6s
                   all       1489      18279          0          0          0          0

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      11/50      13.9G     0.2953     0.3277    0.01788        248        640: 0% ──────────── 0/745  0.3s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      11/50      13.9G        nan        nan        nan        141        640: 49% ━━━━━╸────── 370/745 3.2it/s 1:49<1:59


KeyboardInterrupt: 

In [ ]:
model.train(data="/content/FinalDataSetRevised-1/data.yaml",
            epochs=50, imgsz=640, batch=16,
            amp=False,
            project="/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Five_class_trained_models/results_5Classes_Base_RTDETRL_New_new", #change path here
            name="run1", exist_ok=True)

Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/FinalDataSetRevised-1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/results_RTDETRL_50epochs/run

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       1/50      20.4G     0.3119     0.6258    0.03476         96        640: 100% ━━━━━━━━━━━━ 745/745 2.7it/s 4:38
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 2.9it/s 16.3s
                   all       1489      18279      0.876      0.915      0.932      0.815

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      20.7G     0.2271     0.2898    0.01386        280        640: 0% ──────────── 0/745  0.4s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


       2/50      20.7G     0.2179     0.2821    0.01565        101        640: 100% ━━━━━━━━━━━━ 745/745 2.8it/s 4:30
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 4.2it/s 11.1s
                   all       1489      18279      0.967      0.926      0.971      0.859

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      19.6G     0.1779     0.2523    0.01064        277        640: 0% ──────────── 0/745  0.4s

...

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      49/50      19.6G     0.1101     0.1892   0.007862         66        640: 100% ━━━━━━━━━━━━ 745/745 2.8it/s 4:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 4.2it/s 11.3s
                   all       1489      18279      0.974      0.973      0.985      0.937

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      50/50      19.6G     0.1385     0.2162   0.007813        156        640: 0% ──────────── 0/745  0.4s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)


      50/50      19.6G     0.1112     0.1897    0.00787         57        640: 100% ━━━━━━━━━━━━ 745/745 2.8it/s 4:27
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 47/47 4.2it/s 11.3s
                   all       1489      18279      0.974      0.972      0.985      0.938

50 epochs completed in 3.920 hours.
Optimizer stripped from /content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/results_5Classes_Base_RTDETRL_New/run1/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/results_5Classes_Base_RTDETRL_New/run1/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/results_5Classes_Base_RTDETRL_New/run1/weights/best.pt...
Ultralytics 8.4.164 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
rt-detr-l summary (fused): 315 layers, 31,994,015 parameters, 0 gradients, 10

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7abe5da8d7f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

Hybrid MobileSAM+CLIP (fine tuned)

In [ ]:
!cp "/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Improved_architecture/hybrid_modules.py" /content

In [ ]:
!cp "/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Improved_architecture/hybrid-rtdetr.yaml" /content

In [ ]:
import hybrid_modules                       # must come first: registers the layers with the YAML parser
from ultralytics import RTDETR

model = RTDETR("/content/hybrid-rtdetr.yaml").load("/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Fine_tuned_50epochs_trained_models/results_HybridMobileSAM_Clip_FineTuned/run1/weights/best.pt")
model

Set best params in below cell

In [ ]:
best_params = {
        "optimizer": "NAdam",
        "lr0": 0.0004339992463540169,
        "lrf": 0.1743911646445795,
        "momentum": 0.8534340039797206,
        "weight_decay": 2.7982687195299028e-05,
        "warmup_epochs": 1,
        "box": 6.260865747909289,
        "cls": 0.7800656656035896,
        "dfl": 1.5694067085578194
    }

In [ ]:
model.train(data="/content/FinalDataSetRevised-1/data.yaml",
            epochs=50, imgsz=640, batch=16,
            amp=True,
            project="/content/drive/MyDrive/MS_Passau_Thesis/Newly Modified Hybrid Encoder/Five_class_trained_models/results_5Classes_MobileSAM_Clip_new", #change path here
            name="run1", exist_ok=True)